# 2. Packets: probability, fidelity, delay, and a physical lead density

Read [Chapter 4](../04-packets.md). Synthetic multipliers below test mathematical
identities; they are not claimed to come from electrostatic gates. The later
Gaussian-barrier example uses full physical-model solves. In both sections the
actual `gaussian_packet`, `analyze_packet`, and `reconstruct_lead_packet` are used.
The reference is free propagation between the same planes.

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json
import sys
import numpy as np
import os, tempfile
if "MPLCONFIGDIR" not in os.environ:
    _mpl_cache=tempfile.TemporaryDirectory(prefix="qwave-teaching-mpl-")
    os.environ["MPLCONFIGDIR"]=_mpl_cache.name
import matplotlib.pyplot as plt

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "pyproject.toml").is_file() and (p / "src/qwave").is_dir())
sys.path.insert(0, str(ROOT / "docs/derivations"))
from verification import VerificationBundle
plt.rcParams.update({"font.size": 10, "axes.grid": True, "grid.alpha": .2})

CONFIG = {'seeds': [], 'effective_mass_ratio': 0.067, 'leads_mev': [0.0, 0.0], 'domain_nm': [-40.0, 40.0], 'spatial_points': 321, 'energy_mev': [5.0, 85.0, 401], 'central_energy_mev': 45.0, 'bandwidth_mev': 5.0, 'synthetic': {'global_phase_rad': 0.73, 'constant_attenuation': 1e-05, 'delay_ps': 0.08, 'quadratic_phase_per_mev2': 0.02, 'filter_width_mev': 6.0}, 'delay_search': {'enabled': True, 'min_ps': -0.15, 'max_ps': 0.15, 'grid_points': 121, 'xatol_ps': 1e-09}, 'physical_potential': {'kind': 'gaussian', 'offset_mev': 0.0, 'height_mev': 25.0, 'center_nm': 0.0, 'sigma_nm': 4.0}, 'position_window_nm': [40.0, 1000.0, 961], 'observation_time_ps': 1.0, 'bandwidth_sweep_mev': [2.0, 4.0, 6.0, 8.0, 10.0], 'quadrature_study': {'window_mev': [30.0, 60.0], 'points': [33, 65, 129], 'adaptive_epsabs': 1e-12, 'adaptive_epsrel': 1e-12}, 'approximations': ['finite restricted Gaussian', 'trapezoidal dE measure', 'discrete lead velocity', 'finite right-lead output window; no display renormalization', 'bounded delay search'], 'criteria': {'synthetic_fidelity_atol': 1e-10, 'delay_atol_ps': 2e-07, 'position_norm_atol': 0.002, 'quadrature_ratio_bounds': [3.0, 5.0]}}

output = Path(globals().get("RUN_OUTPUT", ROOT / "results" /
              ("learning-02-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ"))))
bundle = VerificationBundle(output, CONFIG, scope='synthetic identity checks plus a separate Gaussian-barrier model example')
print(json.dumps(CONFIG, indent=2))


In [ ]:
from qwave.scattering import solve_scattering, transmission_phase
from qwave.potentials import sample_potential
from qwave.wavepackets import (gaussian_packet,analyze_packet,DelaySearch,reconstruct_lead_packet,
                              quadrature_weights,HBAR_MEV_PS)
from scipy.integrate import quad
x=np.linspace(*CONFIG["domain_nm"],CONFIG["spatial_points"])
e_low,e_high,e_points=CONFIG["energy_mev"]
energy=np.linspace(e_low,e_high,int(e_points))
center=CONFIG["central_energy_mev"]
sigma=CONFIG["bandwidth_mev"]
free=solve_scattering(x,0*x,energy,effective_mass_ratio=CONFIG["effective_mass_ratio"])
packet=gaussian_packet(energy,central_energy_mev=center,bandwidth_mev=sigma,propagating_bounds_mev=(0.,4*free.hopping_mev))
bundle.json("packet-normalization",packet.normalization)
search=DelaySearch(**CONFIG["delay_search"])
y=energy-center
synth=CONFIG["synthetic"]
multipliers={"reference":np.ones(len(energy)),"global_phase":np.full(len(energy),np.exp(1j*synth["global_phase_rad"])),
 "tiny_constant_output":np.full(len(energy),synth["constant_attenuation"]),
 "delay":np.exp(1j*y*synth["delay_ps"]/HBAR_MEV_PS),
 "quadratic_phase":np.exp(1j*synth["quadratic_phase_per_mev2"]*y*y),
 "amplitude_filter":np.exp(-y*y/(4*synth["filter_width_mev"]**2))}
analyses={name:analyze_packet(energy,packet.amplitude,free.t*mult,free.t,delay=search)
          for name,mult in multipliers.items()}
rows=[{"case":name,**value.summary()} for name,value in analyses.items()]
bundle.table("synthetic-metrics",rows)
beta=synth["quadratic_phase_per_mev2"];q=sigma**2/synth["filter_width_mev"]**2
analytic={"delay":np.exp(-(sigma*synth["delay_ps"]/HBAR_MEV_PS)**2),
          "quadratic_phase":1/np.sqrt(1+4*beta**2*sigma**4),
          "amplitude_filter":np.sqrt(1+q)/(1+q/2)}
error=max(abs(analyses[name].raw_fidelity-value) for name,value in analytic.items())
bundle.json("full-Gaussian-analytic-reference",analytic)
bundle.check("wide-window synthetic Gaussian identities",error<1e-10,measured=error,
 criterion="Full-line formulas agree to 1e-10 for this +/-8 sigma window; omitted tails recorded separately")
bundle.check("reference and global phase",all(abs(analyses[name].raw_fidelity-1)<1e-12 for name in ["reference","global_phase"]),
 measured={name:analyses[name].raw_fidelity for name in ["reference","global_phase"]},criterion="Normalized complex overlap ignores a global phase to 1e-12")
bundle.check("tiny output remains flagged",analyses["tiny_constant_output"].low_transmission and abs(analyses["tiny_constant_output"].raw_fidelity-1)<1e-12,
 measured=analyses["tiny_constant_output"].summary(),criterion="High conditional fidelity must coexist with an explicit low-throughput flag")
bundle.check("positive delay sign and bounded alignment",abs(analyses["delay"].estimated_delay_ps-synth["delay_ps"])<2e-7 and analyses["delay"].aligned_fidelity>1-1e-10,
 measured=analyses["delay"].summary(),criterion="Recover +0.08 ps lag within 2e-7 ps, aligned fidelity >1-1e-10")
bundle.arrays("synthetic-spectra",units={"energy_mev":"meV","incoming_A":"meV^-1/2 complex","free_t":"1 complex",
     "multipliers":"1 complex","delay_scan_ps":"ps","delay_scan_F":"1"},
 energy_mev=energy,incoming_A=packet.amplitude,free_t=free.t,multipliers=np.array(list(multipliers.values())),
 delay_scan_ps=analyses["delay"].scan_delay_ps,delay_scan_F=analyses["delay"].scan_fidelity)
bundle.json("synthetic-spectrum-row-order",list(multipliers))
print(json.dumps(rows,indent=2))


For a separate finite-window quadrature check, compute P5 for a quadratic
phase on $[30,60]$ meV, centered at 45 meV with $\sigma_E=5$ meV. The reference
is an independent adaptive integral of its stated Gaussian numerator and
normalization, not a duplicate packet-analysis function. This window has
non-negligible truncation, so using the full-line identity would be incorrect.

In [ ]:
def density_unnormalized(e):
    return np.exp(-.5*((e-center)/sigma)**2)
q_low,q_high=CONFIG["quadrature_study"]["window_mev"]
quad_args={key:CONFIG["quadrature_study"]["adaptive_"+key] for key in ["epsabs","epsrel"]}
z,zerr=quad(density_unnormalized,q_low,q_high,**quad_args)
real,rerr=quad(lambda e:density_unnormalized(e)*np.cos(beta*(e-center)**2),q_low,q_high,**quad_args)
imag,ierr=quad(lambda e:density_unnormalized(e)*np.sin(beta*(e-center)**2),q_low,q_high,**quad_args)
reference_F=abs((real+1j*imag)/z)**2
convergence=[]
for n in CONFIG["quadrature_study"]["points"]:
    en=np.linspace(q_low,q_high,n)
    p=gaussian_packet(en,central_energy_mev=center,bandwidth_mev=sigma,propagating_bounds_mev=(0.,4*free.hopping_mev))
    value=analyze_packet(en,p.amplitude,np.exp(1j*beta*(en-center)**2),np.ones(n))
    convergence.append({"points":n,"dE_mev":en[1]-en[0],"raw_fidelity":value.raw_fidelity,"absolute_error":abs(value.raw_fidelity-reference_F)})
bundle.table("quadrature-convergence",convergence)
bundle.json("adaptive-reference",{"fidelity":reference_F,"normalization":z,"normalization_error_estimate":zerr,
                                  "real_error_estimate":rerr,"imag_error_estimate":ierr})
errors=np.array([row["absolute_error"] for row in convergence]);ratios=errors[:-1]/errors[1:]
bundle.check("energy quadrature convergence",np.all((ratios>3)&(ratios<5)),measured=ratios,
             criterion="Two halving-step ratios between 3 and 5 for this smooth finite-window integrand")

potential=sample_potential(x,CONFIG["physical_potential"])
device=solve_scattering(x,potential,energy,effective_mass_ratio=CONFIG["effective_mass_ratio"])
metrics=analyze_packet(energy,packet.amplitude,device.t,free.t,delay=search)
p_low,p_high,p_points=CONFIG["position_window_nm"]
position=np.linspace(p_low,p_high,int(p_points))
spatial=dict(k_per_nm=device.k_right_per_nm,velocity_m_s=device.velocity_right_m_s,reference_plane_nm=x[-1],
             time_ps=CONFIG["observation_time_ps"],carrier_energy_mev=center)
outgoing=reconstruct_lead_packet(energy,packet.amplitude*device.t,position,**spatial)
reference=reconstruct_lead_packet(energy,packet.amplitude*free.t,position,**spatial)
# Refine energy at a fixed spatial window to distinguish finite-window loss from quadrature.
en2=np.linspace(e_low,e_high,2*(int(e_points)-1)+1)
dev2=solve_scattering(x,potential,en2,effective_mass_ratio=CONFIG["effective_mass_ratio"])
p2=gaussian_packet(en2,central_energy_mev=center,bandwidth_mev=sigma,propagating_bounds_mev=(0.,4*free.hopping_mev))
out2=reconstruct_lead_packet(en2,p2.amplitude*dev2.t,position,k_per_nm=dev2.k_right_per_nm,
 velocity_m_s=dev2.velocity_right_m_s,reference_plane_nm=x[-1],time_ps=CONFIG["observation_time_ps"],carrier_energy_mev=center)
density_l1=float(np.dot(quadrature_weights(position),abs(out2.density_per_nm-outgoing.density_per_nm)))
norm_error=abs(out2.window_probability-metrics.p_transmitted)
bundle.check("lead Jacobian and window norm",norm_error<.002 and density_l1<.002,measured={"window_norm_error":norm_error,"density_L1_refinement":density_l1},
 criterion="For this late outgoing packet and fixed window, norm and energy-refinement L1 errors <0.002; no spatial renormalization")
bundle.json("physical-metrics",{**metrics.summary(),"outgoing_window_probability":outgoing.window_probability,
    "reference_window_probability":reference.window_probability,"refined_window_probability":out2.window_probability,
    "max_conservation_error":float(max(device.conservation_error)),"density_l1_refinement":density_l1})
bundle.arrays("physical-packet",units={"x_nm":"nm","potential_mev":"meV","energy_mev":"meV","A":"meV^-1/2 complex",
 "t":"1 complex","reference_t":"1 complex","position_nm":"nm","outgoing_density_per_nm":"nm^-1","reference_density_per_nm":"nm^-1",
 "refined_density_per_nm":"nm^-1","refined_energy_mev":"meV","refined_A":"meV^-1/2 complex","refined_t":"1 complex"},
 x_nm=x,potential_mev=potential,energy_mev=energy,A=packet.amplitude,t=device.t,reference_t=free.t,position_nm=position,
 outgoing_density_per_nm=outgoing.density_per_nm,reference_density_per_nm=reference.density_per_nm,
 refined_density_per_nm=out2.density_per_nm,refined_energy_mev=en2,refined_A=p2.amplitude,refined_t=dev2.t)
print(json.dumps(metrics.summary(),indent=2))


In [ ]:
bandwidth_rows=[]
for width in CONFIG["bandwidth_sweep_mev"]:
    p=gaussian_packet(energy,central_energy_mev=center,bandwidth_mev=width,propagating_bounds_mev=(0.,4*free.hopping_mev))
    m=analyze_packet(energy,p.amplitude,device.t,free.t,delay=search)
    bandwidth_rows.append({"bandwidth_mev":width,"P_trans":m.p_transmitted,"F_raw":m.raw_fidelity,"F_aligned":m.aligned_fidelity,
                          "delay_ps":m.estimated_delay_ps,"untruncated_mass_in_window":p.normalization["untruncated_gaussian_mass_in_window"]})
bundle.table("bandwidth",bandwidth_rows)
fig,axs=plt.subplots(2,2,figsize=(10,7),layout="constrained")
axs[0,0].plot(energy,abs(packet.amplitude)**2,label="Incident density")
axs[0,0].plot(energy,abs(packet.amplitude*device.t)**2,label="Transmitted (integral = P)")
axs[0,0].set(xlabel="Energy (meV)",ylabel="Energy probability density (1/meV)");axs[0,0].legend()
axs[0,1].plot(position,reference.density_per_nm,label="Free reference output")
axs[0,1].plot(position,outgoing.density_per_nm,label="Barrier output")
axs[0,1].set(xlabel="Right-lead position (nm)",ylabel="Position probability density (1/nm)");axs[0,1].legend()
axs[1,0].plot(energy,transmission_phase(device.t*free.t.conj()))
axs[1,0].set(xlabel="Energy (meV)",ylabel="Unwrapped phase difference (rad)")
for key in ["P_trans","F_raw","F_aligned"]:
    axs[1,1].plot([row["bandwidth_mev"] for row in bandwidth_rows],[row[key] for row in bandwidth_rows],"o-",label=key)
axs[1,1].set(xlabel="Untruncated bandwidth sigma (meV)",ylabel="Probability or conditional fidelity (1)");axs[1,1].legend()
bundle.figure(fig,"packets","Gaussian-barrier model, mass ratio 0.067. Position curves are outgoing right-lead components at 1 ps, not interior states or trajectories. Separate spectral throughput and conditional fidelity; finite windows are saved.")
fig,axs=plt.subplots(1,2,figsize=(10,3.8),layout="constrained")
axs[0].plot(analyses["delay"].scan_delay_ps,analyses["delay"].scan_fidelity)
axs[0].axvline(synth["delay_ps"],ls="--",label="Specified synthetic lag")
axs[0].set(xlabel="Alignment lag (ps)",ylabel="Conditional fidelity (1)");axs[0].legend()
axs[1].loglog([row["dE_mev"] for row in convergence],errors,"o-")
axs[1].set(xlabel="Energy spacing (meV)",ylabel="Raw fidelity error (1)")
bundle.figure(fig,"packet-checks","Synthetic pure-delay sign check and separate finite-window quadratic-phase quadrature convergence. These multipliers are mathematical benchmarks, not simulated gate designs.")
bundle.finish()
